In [1]:
# Imports + Dataset Load + Basic Check
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import joblib
import os

df = pd.read_csv("../data/raw/creditcard.csv")

print("Shape:", df.shape)
print(df["Class"].value_counts())
print("Fraud %:", df["Class"].mean()*100)

Shape: (284807, 31)
Class
0    284315
1       492
Name: count, dtype: int64
Fraud %: 0.1727485630620034


In [2]:
# X, y + Scaling
x = df.drop("Class", axis=1).copy()
y = df["Class"].copy()

scaler = StandardScaler()
x_scaled = pd.DataFrame(scaler.fit_transform(x), columns=x.columns)

In [3]:
# Train/Test Split
x_train, x_test, y_train, y_test = train_test_split(
    x_scaled, y, test_size=0.2, stratify=y, random_state=42
)

print("Shapes:", x_train.shape, x_test.shape)

Shapes: (227845, 30) (56962, 30)


In [4]:
# Save processed data
os.makedirs("models", exist_ok=True)
os.makedirs("data/processed", exist_ok=True)
os.makedirs("reports", exist_ok=True)

joblib.dump(scaler, "models/scaler_deep.pkl")
joblib.dump(x_train, "data/processed/X_train_deep.pkl")
joblib.dump(x_test, "data/processed/X_test_deep.pkl")
joblib.dump(y_train, "data/processed/y_train_deep.pkl")
joblib.dump(y_test, "data/processed/y_test_deep.pkl")

print("Saved deep learning preprocessing files")

Saved deep learning preprocessing files


In [5]:
# TensorFlow imports + Load data
from tensorflow import keras
from tensorflow.keras import layers

x_train = joblib.load("data/processed/X_train_deep.pkl")
y_train = joblib.load("data/processed/y_train_deep.pkl")
x_test = joblib.load("data/processed/X_test_deep.pkl")
y_test = joblib.load("data/processed/y_test_deep.pkl")

x_train_normal = x_train[y_train == 0]
input_dim = x_train_normal.shape[1]

print("Normal train shape:", x_train_normal.shape)

Normal train shape: (227451, 30)


In [6]:
# Build Autoencoder Model
input_layer = keras.Input(shape=(input_dim,))

encoded = layers.Dense(32, activation="relu")(input_layer)
encoded = layers.Dense(16, activation="relu")(encoded)
encoded = layers.Dense(8, activation="relu")(encoded)

decoded = layers.Dense(16, activation="relu")(encoded)
decoded = layers.Dense(32, activation="relu")(decoded)
decoded = layers.Dense(input_dim, activation="linear")(decoded)

autoencoder = keras.Model(inputs=input_layer, outputs=decoded)
autoencoder.compile(optimizer="adam", loss="mse")

autoencoder.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 30)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │           992 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 8)              │           136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 16)             │           144 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 32)             │           544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 30)             │           990 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,334 (13.02 KB)

 Trainable params: 3,334 (13.02 KB)

 Non-trainable params: 0 (0.00 B)

In [7]:
# Train Autoencoder
history = autoencoder.fit(
    x_train_normal,
    x_train_normal,
    epochs=30,
    batch_size=2048,
    validation_split=0.1,
    callbacks=[
        keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=3, restore_best_weights=True
        )
    ],
    verbose=2
)

autoencoder.save("models/autoencoder.keras")
print("Autoencoder saved")

Epoch 1/30
100/100 - 4s - 41ms/step - loss: 0.9081 - val_loss: 0.8115
Epoch 2/30
100/100 - 1s - 10ms/step - loss: 0.7546 - val_loss: 0.6741
Epoch 3/30
100/100 - 1s - 12ms/step - loss: 0.6431 - val_loss: 0.5860
Epoch 4/30
100/100 - 1s - 12ms/step - loss: 0.5710 - val_loss: 0.5340
Epoch 5/30
100/100 - 1s - 7ms/step - loss: 0.5252 - val_loss: 0.5010
Epoch 6/30
100/100 - 1s - 13ms/step - loss: 0.4947 - val_loss: 0.4736
Epoch 7/30
100/100 - 1s - 7ms/step - loss: 0.4700 - val_loss: 0.4508
Epoch 8/30
100/100 - 1s - 13ms/step - loss: 0.4498 - val_loss: 0.4345
Epoch 9/30
100/100 - 1s - 7ms/step - loss: 0.4327 - val_loss: 0.4186
Epoch 10/30
100/100 - 1s - 8ms/step - loss: 0.4184 - val_loss: 0.4051
Epoch 11/30
100/100 - 1s - 13ms/step - loss: 0.4066 - val_loss: 0.3943
Epoch 12/30
100/100 - 1s - 8ms/step - loss: 0.3963 - val_loss: 0.3856
Epoch 13/30
100/100 - 1s - 8ms/step - loss: 0.3876 - val_loss: 0.3784
Epoch 14/30
100/100 - 1s - 8ms/step - loss: 0.3807 - val_loss: 0.3715
Epoch 15/30
100/100 - 

In [8]:
# Threshold + Evaluation
from sklearn.metrics import classification_report, roc_auc_score

x_test_pred = autoencoder.predict(x_test, batch_size=2048)
mse_test = np.mean(np.square(x_test - x_test_pred), axis=1)

x_val_pred = autoencoder.predict(x_train_normal, batch_size=2048)
mse_val = np.mean(np.square(x_train_normal - x_val_pred), axis=1)

threshold = np.percentile(mse_val, 99.5)
y_pred_ae = (mse_test > threshold).astype(int)

print("Threshold:", threshold)
print(classification_report(y_test, y_pred_ae))
print("AE ROC-AUC:", roc_auc_score(y_test, mse_test))

28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step  
112/112 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 
Threshold: 1.943107716921141
              precision    recall  f1-score   support

           0       1.00      0.99      1.00     56864
           1       0.22      0.82      0.34        98

    accuracy                           0.99     56962
   macro avg       0.61      0.91      0.67     56962
weighted avg       1.00      0.99      1.00     56962

AE ROC-AUC: 0.9486203386813362


In [9]:
# ANN model training
from tensorflow.keras import models, optimizers
from sklearn.utils import class_weight

cw = class_weight.compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train
)
cw_dict = {0: cw[0], 1: cw[1]}
print("Class weights:", cw_dict)

model_ann = models.Sequential([
    layers.Input(shape=(x_train.shape[1],)),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.2),
    layers.Dense(32, activation="relu"),
    layers.Dense(1, activation="sigmoid")
])

model_ann.compile(
    optimizer=optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=["AUC"]
)

history_ann = model_ann.fit(
    x_train, y_train,
    epochs=30,
    batch_size=1024,
    validation_split=0.1,
    class_weight=cw_dict,
    callbacks=[
        keras.callbacks.EarlyStopping(
            monitor="val_AUC", patience=4, mode="max", restore_best_weights=True
        )
    ],
    verbose=2
)

model_ann.save("models/ann_classifier.keras")
print("ANN saved")

Class weights: {0: np.float64(0.5008661206149896), 1: np.float64(289.14340101522845)}
Epoch 1/30
201/201 - 7s - 32ms/step - AUC: 0.9192 - loss: 0.3236 - val_AUC: 0.9928 - val_loss: 0.1488
Epoch 2/30
201/201 - 2s - 12ms/step - AUC: 0.9789 - loss: 0.1673 - val_AUC: 0.9963 - val_loss: 0.1448
Epoch 3/30
201/201 - 3s - 13ms/step - AUC: 0.9849 - loss: 0.1469 - val_AUC: 0.9967 - val_loss: 0.0743
Epoch 4/30
201/201 - 3s - 14ms/step - AUC: 0.9894 - loss: 0.1221 - val_AUC: 0.9967 - val_loss: 0.0880
Epoch 5/30
201/201 - 3s - 13ms/step - AUC: 0.9891 - loss: 0.1231 - val_AUC: 0.9956 - val_loss: 0.0683
Epoch 6/30
201/201 - 2s - 12ms/step - AUC: 0.9927 - loss: 0.1055 - val_AUC: 0.9985 - val_loss: 0.0734
Epoch 7/30
201/201 - 2s - 12ms/step - AUC: 0.9912 - loss: 0.1174 - val_AUC: 0.9976 - val_loss: 0.0828
Epoch 8/30
201/201 - 2s - 12ms/step - AUC: 0.9913 - loss: 0.1101 - val_AUC: 0.9985 - val_loss: 0.1035
Epoch 9/30
201/201 - 2s - 12ms/step - AUC: 0.9930 - loss: 0.1041 - val_AUC: 0.9962 - val_loss: 0.0

In [10]:
# ANN Evaluation
from sklearn.metrics import roc_auc_score, classification_report

y_proba_ann = model_ann.predict(x_test).ravel()
y_pred_ann = (y_proba_ann >= 0.5).astype(int)

print("ANN ROC-AUC:", roc_auc_score(y_test, y_proba_ann))
print(classification_report(y_test, y_pred_ann))

1781/1781 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step
ANN ROC-AUC: 0.9804085903494768
              precision    recall  f1-score   support

           0       1.00      0.95      0.98     56864
           1       0.03      0.94      0.07        98

    accuracy                           0.95     56962
   macro avg       0.52      0.95      0.52     56962
weighted avg       1.00      0.95      0.98     56962



In [11]:
# Create sequences
x_np = x_train.values
y_np = y_train.values

def create_sequences(xarr, yarr, window=10, step=1):
    xs, ys = [], []
    for i in range(0, len(xarr) - window, step):
        seq = xarr[i:i+window]
        label = 1 if yarr[i:i+window].sum() > 0 else 0
        xs.append(seq)
        ys.append(label)
    return np.array(xs), np.array(ys)

window = 10
x_seq, y_seq = create_sequences(x_np, y_np, window=window)

from sklearn.model_selection import train_test_split
x_seq_train, x_seq_val, y_seq_train, y_seq_val = train_test_split(
    x_seq, y_seq, test_size=0.2, random_state=42, stratify=y_seq
)

print("Seq shape:", x_seq_train.shape, y_seq_train.shape)

Seq shape: (182268, 10, 30) (182268,)


In [13]:
# LSTM model
timesteps = x_seq_train.shape[1]
features = x_seq_train.shape[2]

lstm_model = models.Sequential([
    layers.Input(shape=(timesteps, features)),
    layers.LSTM(64, return_sequences=True),
    layers.Dropout(0.3),
    layers.LSTM(32),
    layers.Dense(32, activation="relu"),
    layers.Dense(1, activation="sigmoid")
])

lstm_model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["AUC"])

lstm_history = lstm_model.fit(
    x_seq_train, y_seq_train,
    validation_data=(x_seq_val, y_seq_val),
    epochs=20,
    batch_size=256,
    callbacks=[
        keras.callbacks.EarlyStopping(
            monitor="val_AUC", patience=4, mode="max", restore_best_weights=True
        )
    ],
    verbose=2
)

y_proba_lstm = lstm_model.predict(x_seq_val).ravel()
print("LSTM ROC-AUC:", roc_auc_score(y_seq_val, y_proba_lstm))

Epoch 1/20
712/712 - 30s - 42ms/step - AUC: 0.9348 - loss: 0.0445 - val_AUC: 0.9553 - val_loss: 0.0244
Epoch 2/20
712/712 - 24s - 34ms/step - AUC: 0.9663 - loss: 0.0182 - val_AUC: 0.9734 - val_loss: 0.0156
Epoch 3/20
712/712 - 24s - 33ms/step - AUC: 0.9816 - loss: 0.0109 - val_AUC: 0.9827 - val_loss: 0.0103
Epoch 4/20
712/712 - 24s - 34ms/step - AUC: 0.9916 - loss: 0.0066 - val_AUC: 0.9863 - val_loss: 0.0070
Epoch 5/20
712/712 - 38s - 53ms/step - AUC: 0.9967 - loss: 0.0042 - val_AUC: 0.9903 - val_loss: 0.0058
Epoch 6/20
712/712 - 22s - 30ms/step - AUC: 0.9977 - loss: 0.0028 - val_AUC: 0.9948 - val_loss: 0.0037
Epoch 7/20
712/712 - 24s - 33ms/step - AUC: 0.9985 - loss: 0.0021 - val_AUC: 0.9935 - val_loss: 0.0050
Epoch 8/20
712/712 - 39s - 55ms/step - AUC: 0.9988 - loss: 0.0017 - val_AUC: 0.9922 - val_loss: 0.0047
Epoch 9/20
712/712 - 22s - 31ms/step - AUC: 0.9995 - loss: 0.0013 - val_AUC: 0.9935 - val_loss: 0.0041
Epoch 10/20
712/712 - 27s - 38ms/step - AUC: 0.9989 - loss: 0.0014 - val_

In [14]:
# Compare & Save best
from sklearn.metrics import roc_auc_score
import json

y_proba_ann = model_ann.predict(x_test).ravel()
ann_auc = roc_auc_score(y_test, y_proba_ann)

ae_auc = roc_auc_score(y_test, mse_test)

print("ANN AUC:", ann_auc)
print("AE AUC:", ae_auc)

best_name = "ann" if ann_auc >= ae_auc else "autoencoder"
print("Best model:", best_name)

if best_name == "ann":
    model_ann.save("models/best_model_ann.keras")
else:
    autoencoder.save("models/best_model_autoencoder.keras")

report = {
    "ANN_AUC": float(ann_auc),
    "AE_AUC": float(ae_auc),
    "best_model": best_name
}

with open("reports/deep_learning_report.json", "w") as f:
    json.dump(report, f, indent=4)

print("Saved report")

1781/1781 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step
ANN AUC: 0.9804085903494768
AE AUC: 0.9486203386813362
Best model: ann
Saved report
